# 🌦️ Weather Forecasting — Model Training Notebook

**What this notebook does (in plain English)**

> We have 11 years of *hourly* weather (2006–2016). We teach a computer to look at the weather **right now and
> in the recent past** and predict what the weather will be **1 to 72 hours from now**.
> We then check — honestly, on years the computer has never seen — that it really is better than simple guesses.
> Finally we save the trained models into the `models/` folder so the Streamlit app (`main.py`) can use them.

**How to run:** `Kernel → Restart & Run All`. It takes a few minutes. Nothing else to do.

### The big picture
```
raw CSV ─► 1 Clean ─► 2 Explore ─► 3 Build features ─► 4 Split by time ─► 5 Train models
                                                                              │
   main.py (Streamlit app) ◄─ 9 Save to /models ◄─ 8 Explain ◄─ 7 Uncertainty ◄─ 6 Test on unseen years
```

### Words you will see
| Word | Meaning in one line |
|---|---|
| **Horizon** | How many hours into the future we predict (`+24 h` = tomorrow, same time). |
| **Feature** | One number the model looks at (e.g. "temperature 6 hours ago"). |
| **Lag** | A past value (`temp_c_lag24` = temperature 24 hours ago). |
| **Baseline** | A dumb-but-honest guess we must beat, otherwise our model is pointless. |
| **MAE** | Mean Absolute Error = "on average we are off by this much" (in °C, km/h, …). |
| **Skill** | How much better than the baseline we are, e.g. `40 %` = 40 % smaller error. |
| **Data leakage** | Accidentally letting the model peek at the future. Makes results look great and fake. |
| **Prediction interval** | A range (“between 12 and 16 °C”) that the truth falls into ~80 % of the time. |

In [1]:
import json, platform, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sklearn
from IPython.display import display

from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, log_loss, classification_report
from sklearn.inspection import permutation_importance

import weather_core as wc          # our shared helper file (cleaning, features, forecasting)

ModuleNotFoundError: No module named 'seaborn'

In [1]:


warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})

# ───────────── Settings you may change ─────────────
DATA_PATH = Path("data/weatherHistory.csv")
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)
RANDOM_STATE = 42

# Time-based split (NEVER shuffle a time series!)
TRAIN_END = pd.Timestamp("2014-01-01", tz="UTC")   # train : 2006 - 2013
VALID_END = pd.Timestamp("2015-01-01", tz="UTC")   # valid : 2014     (choose #trees + calibrate uncertainty)
                                                   # test  : 2015-16  (used ONCE, for the final honest score)
ROWS_PER_ORIGIN_REG = 5      # random horizons drawn per start time (regression)
ROWS_PER_ORIGIN_CLF = 3      # ... (sky-condition classifier)
MAX_ITER_REG = 300           # maximum boosting rounds (best number is picked automatically)
MAX_ITER_CLF = 150
HGB_PARAMS = dict(learning_rate=0.08, max_leaf_nodes=48, min_samples_leaf=200,
                  l2_regularization=1.0, early_stopping=False, random_state=RANDOM_STATE)

MODELS = ["Persistence", "Same time yesterday", "Ridge (linear)", "Boosting (ours)"]
MODEL_STYLE = {"Persistence": dict(color="#adb5bd", ls="--", lw=1.6),
               "Same time yesterday": dict(color="#c9a66b", ls="--", lw=1.6),
               "Ridge (linear)": dict(color="#4c8dd0", ls="", lw=0, marker="o", ms=6),
               "Boosting (ours)": dict(color="#d9480f", ls="-", lw=2.6)}
print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

NameError: name 'warnings' is not defined

---
## 1 · Load & clean the data

Real data is messy. Before any machine learning we look for problems and fix them.
The cleaning code lives in `weather_core.load_and_clean()` (so the app cleans uploaded files **exactly** the same way).

In [ ]:
df, report = wc.load_and_clean(DATA_PATH)

issues = pd.DataFrame([
    ("File not sorted by time",                         "yes" if report["was_unsorted"] else "no", "sorted chronologically"),
    ("Duplicate timestamps",                            report["duplicate_timestamps_removed"],    "kept the first row of each"),
    ("Pressure = 0 hPa (physically impossible)",        report["pressure_invalid"],                "marked missing → filled by interpolation (gaps are ≤ 2 h)"),
    ("Humidity = 0 % (impossible here)",                report["humidity_invalid"],                "marked missing → interpolated"),
    ("Visibility = 0 km (mostly under CLEAR sky)",      report["visibility_invalid"],              "sensor placeholder → marked missing → interpolated"),
    ("Missing hours in the timeline",                   report["missing_hours_filled"],            "re-created so every hour exists, then interpolated"),
    ("Column 'Loud Cover'",                             "all 0",                                   "dropped - contains no information"),
    ("Column 'Daily Summary'",                          "214 values",                              "dropped - DATA LEAK: it describes the whole day, including hours still in the future"),
    ("Column 'Precip Type'",                            "rain/snow",                               "dropped - it is simply 'snow if temperature < 0 °C', so it adds nothing"),
], columns=["Problem found", "How many", "What we did"])
display(issues)

print(f"\nClean table: {df.shape[0]:,} hourly rows from {report['start'][:10]} to {report['end'][:10]}  (all times in UTC)")
display(df.head(3))

**Why does `Daily Summary` matter so much?** Imagine predicting the weather at 10:00 while the model is secretly told
"today it rains in the afternoon". Results would look amazing but be *impossible* in real life. Spotting this
kind of trap is a big part of doing forecasting properly.

**Simplifying the sky label.** The raw `Summary` column has 27 messy labels ("Breezy and Mostly Cloudy" …).
We reduce them to 5 clear classes: *Clear, Partly Cloudy, Mostly Cloudy, Overcast, Foggy*.

---
## 2 · Explore the data (EDA)

Before modelling, *look* at the data. What patterns can a model learn?

In [ ]:
cols = list(wc.TARGETS)
summary = df[cols].describe().T[["mean", "std", "min", "50%", "max"]].round(2)
summary.insert(0, "unit", [wc.TARGETS[c]["unit"] for c in cols])
summary.index = [wc.TARGETS[c]["label"] for c in cols]
display(summary)

In [ ]:
loc = wc.to_local(df.index)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2), gridspec_kw={"width_ratios": [2.2, 1]})

daily = df["temp_c"].resample("D").mean()
ax[0].plot(daily.index, daily, color="#f1b58f", lw=0.7, label="daily average")
ax[0].plot(daily.index, daily.rolling(30, center=True).mean(), color="#d9480f", lw=2, label="30-day average")
ax[0].set_title("Temperature 2006-2016: a strong yearly cycle"); ax[0].set_ylabel("°C"); ax[0].legend(frameon=False)

months = pd.Series(loc.month, index=df.index)
sns.boxplot(x=months, y=df["temp_c"], ax=ax[1], color="#f3c9ae", fliersize=0, linewidth=1)
ax[1].set_title("Temperature by month"); ax[1].set_xlabel("month"); ax[1].set_ylabel("")
plt.tight_layout(); plt.show()

In [ ]:
pivot = (df.assign(month=loc.month, hour=loc.hour)
           .pivot_table(index="hour", columns="month", values="temp_c", aggfunc="mean"))
fig, ax = plt.subplots(figsize=(11, 4.6))
sns.heatmap(pivot, cmap="RdYlBu_r", ax=ax, cbar_kws={"label": "average °C"})
ax.invert_yaxis(); ax.set_title("Average temperature by hour of day (rows) and month (columns)")
ax.set_xlabel("month"); ax.set_ylabel("local hour")
plt.tight_layout(); plt.show()

**What to notice:** temperature has *two* rhythms — a **daily** one (warm afternoons, cold nights) and a **yearly** one
(hot summers, cold winters). That is why our features include the hour of day and the day of year.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.6), gridspec_kw={"width_ratios": [1, 1.2]})

corr = df[cols].corr()
corr.index = corr.columns = [wc.TARGETS[c]["label"] for c in cols]
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax[0], cbar=False)
ax[0].set_title("How the variables move together")

mix = pd.crosstab(loc.month, df["cond"], normalize="index")[wc.COND_CLASSES]
mix.plot(kind="bar", stacked=True, ax=ax[1], color=[wc.COND_COLORS[c] for c in mix.columns], width=0.85)
ax[1].set_title("Sky condition mix by month (fog loves winter)"); ax[1].set_xlabel("month"); ax[1].set_ylabel("share of hours")
ax[1].legend(frameon=False, fontsize=8, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.15))
plt.tight_layout(); plt.show()

**What to notice:** humidity and temperature are *negatively* related (warm air holds more water, so relative humidity drops in the afternoon);
visibility falls when humidity is high (fog). The model will discover such relationships by itself.

---
## 3 · Frame the problem & build features

### How we forecast: “one smart model that knows how far ahead it is looking”
For every start time `t` (say, 08:00 today) we want the value at `t + h` for any `h` from 1 to 72 hours.

Three design choices — each is simple, and each makes the model better:

1. **Direct forecasting, not step-by-step.** We do not predict 1 h, then use that to predict 2 h, then 3 h …
   (small mistakes pile up). Instead the model jumps straight to the target hour.
2. **`h` is an input.** One model per weather variable receives “how many hours ahead” as a feature (plus the clock time of the target hour),
   so it can draw a smooth curve for *every* hour and knows that 3 pm is warmer than 3 am.
3. **Predict the change, not the value.** The model learns *“how much will the temperature change from now?”*
   (target = `value(t+h) − value(t)`), then we add that to today's value. It is much easier to learn “+3 °C” than “18.4 °C”.

### The features (what the model looks at)
| Family | Examples | Why it helps |
|---|---|---|
| Current conditions | temperature, humidity, wind, pressure, visibility | the starting point |
| History (lags) | value 1, 2, 3, 6, 12, 24, 48, 72 h ago | "what was it doing recently?" |
| Trends | 3-hour pressure change, 24-hour temperature change | falling pressure ⇒ storms; trend ⇒ direction |
| Rolling summaries | last-24 h min/max/std, last-7-days average | is today unusual for this week? |
| Physics | dew point, *temperature − dew point* | tiny gap ⇒ air is nearly saturated ⇒ fog/cloud |
| Wind as a vector | `u`, `v`, `sin`, `cos` of direction | 359° and 1° are neighbours, not opposites |
| Clock (cyclic) | hour-of-day, day-of-year as `sin`/`cos` | 23:00 is next to 00:00 |

In [ ]:
feat = wc.build_base_features(df)              # one row per hour, only PAST + PRESENT information
X_base = feat.to_numpy(dtype=float)
base_columns = list(feat.columns)
model_columns = base_columns + wc.H_COLS       # + "how far ahead" columns added later
ts = df.index

families = (pd.Series([wc.feature_group(c) for c in model_columns])
              .value_counts().rename("number of features").to_frame())
print(f"{len(model_columns)} features in total")
display(families)
display(feat.iloc[[1000]].T.head(12).rename(columns={feat.index[1000]: "example row (one hour)"}).round(2))

### 🔒 Safety check: prove there is no data leakage
We take the data, **destroy everything after time `t`** (add +1000 to every value), rebuild all features, and check that
the features **up to `t` did not change**. If any feature secretly used future data, this test would fail.

In [ ]:
t = 50_000
value_cols = ["temp_c", "feels_c", "hum", "wind_kmh", "vis_km", "pres_mb"]
corrupted = df.copy()
corrupted.loc[corrupted.index[t + 1:], value_cols] += 1000          # wreck the future

feat_bad = wc.build_base_features(corrupted)
identical = np.allclose(feat.iloc[: t + 1].to_numpy(float), feat_bad.iloc[: t + 1].to_numpy(float), equal_nan=True)
assert identical, "LEAK! a feature uses future data"
print("✅ Passed: features at time t only use information available at time t.")

---
## 4 · Split the data **by time** (never randomly)

If we shuffled the rows, the model would train on 10:00 and be tested on 11:00 of the same day — nearly identical hours — and look
brilliant for no good reason. In real life we predict the *future*, so the test must be the *future* too:

| Part | Years | Used for |
|---|---|---|
| **Train** | 2006 – 2013 | the model learns from this |
| **Validation** | 2014 | pick the number of boosting rounds, calibrate the uncertainty bands |
| **Test** | 2015 – 2016 | final exam — touched **once** at the end |

We also leave a 72-hour gap at the end of train and validation so that no training answer peeks into the next period.

In [ ]:
origins = wc.valid_origins(len(df))                     # rows that can be forecast starts
gap = pd.Timedelta(hours=wc.MAX_HORIZON)
train_o = origins[ts[origins] < TRAIN_END - gap]
valid_o = origins[(ts[origins] >= TRAIN_END) & (ts[origins] < VALID_END - gap)]
test_o  = origins[ts[origins] >= VALID_END]

fig, ax = plt.subplots(figsize=(11, 1.6))
for name, idx, colr in [("Train", train_o, "#4c8dd0"), ("Valid.", valid_o, "#f59f00"), ("Test", test_o, "#d9480f")]:
    ax.barh(0, (ts[idx[-1]] - ts[idx[0]]).days, left=ts[idx[0]], color=colr, height=0.5)
    ax.text(ts[idx[0]] + (ts[idx[-1]] - ts[idx[0]]) / 2, 0, f"{name}\n{len(idx):,} start times", ha="center", va="center", color="white", fontsize=9, fontweight="bold")
ax.set_yticks([]); ax.set_title("Chronological split")
plt.tight_layout(); plt.show()

---
## 5 · Train the models

### 5a · Build the training table
Each training row is a pair *(start time, horizon)*. For every start time in 2006–2013 we draw a few random horizons between 1 and 72 hours
and record the true change that happened. That gives the model hundreds of thousands of examples of “given this weather now, this is what happened *h* hours later”.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
Y = {v: df[v].to_numpy(float) for v in wc.TARGETS}                        # full series per variable
CUR = {v: feat[v].to_numpy(float) for v in wc.TARGETS}                    # 'value right now' per start time
COND = df["cond_code"].to_numpy(float)

o_tr, h_tr = wc.sample_pairs(train_o, ROWS_PER_ORIGIN_REG, rng)
o_va, h_va = wc.sample_pairs(valid_o, 3, rng)
X_tr = wc.horizon_matrix(X_base[o_tr], ts[o_tr], h_tr)
X_va = wc.horizon_matrix(X_base[o_va], ts[o_va], h_va)
assert X_tr.shape[1] == len(model_columns)
print(f"training rows  : {X_tr.shape[0]:,}  x {X_tr.shape[1]} features")
print(f"validation rows: {X_va.shape[0]:,}")

### 5b · Gradient boosting — the “full power” model
**Gradient boosting** builds hundreds of small decision trees one after another; each new tree focuses on fixing the mistakes the previous ones made.
It is a very strong all-round choice for tabular problems like this one, and it needs no feature scaling and copes with missing values.
(We use scikit-learn's `HistGradientBoosting`, which is the same algorithm family as LightGBM/XGBoost — so no extra installation.)

**One extra trick — scaling the target.** After 1 hour the temperature typically moves about 1 °C, after 24 hours about 5 °C. If we trained on raw changes, the large
long-range swings would dominate the learning and the first hours would be neglected. So we divide every change by the *typical change at that horizon*
(`wc.horizon_sigma`), train on that, and multiply back when predicting (`wc.ScaledDeltaModel`). Same model, fairer training.

**How many trees?** Too few = underfit, too many = memorise the training years. We train up to 300 rounds, watch the error on the **validation year** after every round,
and keep the best round count.

In [ ]:
regressors, best_iter, val_curve, sigmas = {}, {}, {}, {}
H_IDX = model_columns.index("h")                       # which input column holds the horizon
t_start = time.time()
for var in wc.TARGETS:
    # target = CHANGE from now (value at t+h minus value at t)
    d_tr = Y[var][o_tr + h_tr] - CUR[var][o_tr]
    d_va = Y[var][o_va + h_va] - CUR[var][o_va]
    ok_tr, ok_va = ~np.isnan(d_tr), ~np.isnan(d_va)

    # scale the change by "how big is a typical change after h hours" so short horizons are learned as carefully as long ones
    sig = wc.horizon_sigma(d_tr[ok_tr], h_tr[ok_tr])
    s_tr, s_va = sig[h_tr - 1], sig[h_va - 1]

    model = HistGradientBoostingRegressor(max_iter=MAX_ITER_REG, **HGB_PARAMS).fit(X_tr[ok_tr], d_tr[ok_tr] / s_tr[ok_tr])
    # validation error in REAL units (multiply the scaled prediction back by sigma)
    curve = np.array([np.abs(p * s_va[ok_va] - d_va[ok_va]).mean() for p in model.staged_predict(X_va[ok_va])])
    best = int(curve.argmin()) + 1
    if best < 0.9 * MAX_ITER_REG:                      # re-train with exactly the best number of rounds
        model = HistGradientBoostingRegressor(max_iter=best, **HGB_PARAMS).fit(X_tr[ok_tr], d_tr[ok_tr] / s_tr[ok_tr])
    regressors[var] = wc.ScaledDeltaModel(model, sig, H_IDX)      # wrapper: predict() returns real-unit changes
    best_iter[var], val_curve[var], sigmas[var] = best, curve, sig
    print(f"{wc.TARGETS[var]['label']:<12} best rounds = {best:>3}   validation MAE = {curve[best-1]:.3f} {wc.TARGETS[var]['unit']}"
          f"   (guessing 'no change': {np.abs(d_va[ok_va]).mean():.3f})   [{time.time()-t_start:.0f}s]")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 6), sharex=False)
for ax, var in zip(axes.ravel(), wc.TARGETS):
    c = val_curve[var]
    ax.plot(np.arange(1, len(c) + 1), c, color=wc.TARGETS[var]["color"], lw=2)
    ax.axvline(best_iter[var], color="k", ls=":", lw=1)
    ax.set_title(f"{wc.TARGETS[var]['label']}  (best = {best_iter[var]} rounds)", fontsize=10)
    ax.set_xlabel("boosting rounds (trees)"); ax.set_ylabel("validation MAE")
plt.suptitle("Learning curves: error on the validation year as trees are added", y=1.02)
plt.tight_layout(); plt.show()

**How to read this:** the error drops quickly and then flattens. If it started going *up* again we would be overfitting — we stop at the lowest point (dotted line).

### 5c · A second model: the sky-condition classifier
Besides numbers we also predict the **sky condition** (Clear / Partly / Mostly Cloudy / Overcast / Foggy). This model outputs a *probability* for each class
(“60 % overcast, 25 % mostly cloudy …”), which the app turns into a chance-of-fog chart.

> **Why a different sampling?** The sky rarely changes within a few hours, so “just repeat the current sky” is a strong baseline at short range. To beat it, the classifier sees *more* short-horizon examples (about 60 % of its training rows are within +12 h) than the regressors do.

In [ ]:
o_c, h_c = wc.sample_pairs(train_o, ROWS_PER_ORIGIN_CLF, rng, log_uniform=True)   # more short-horizon examples
o_cv, h_cv = wc.sample_pairs(valid_o, 2, rng, log_uniform=True)
X_c  = wc.horizon_matrix(X_base[o_c],  ts[o_c],  h_c)
X_cv = wc.horizon_matrix(X_base[o_cv], ts[o_cv], h_cv)
y_c, y_cv = COND[o_c + h_c], COND[o_cv + h_cv]
ok_c, ok_cv = ~np.isnan(y_c), ~np.isnan(y_cv)
CLF_PARAMS = dict(HGB_PARAMS, learning_rate=0.1, max_leaf_nodes=31)

clf = HistGradientBoostingClassifier(max_iter=MAX_ITER_CLF, **CLF_PARAMS).fit(X_c[ok_c], y_c[ok_c].astype(int))
clf_curve = np.array([log_loss(y_cv[ok_cv].astype(int), p, labels=clf.classes_) for p in clf.staged_predict_proba(X_cv[ok_cv])])
clf_best = int(clf_curve.argmin()) + 1
if clf_best < 0.9 * MAX_ITER_CLF:
    clf = HistGradientBoostingClassifier(max_iter=clf_best, **CLF_PARAMS).fit(X_c[ok_c], y_c[ok_c].astype(int))
print(f"classifier: best rounds = {clf_best}, validation log-loss = {clf_curve[clf_best-1]:.3f} "
      f"(a model that always predicts the class mix would score {log_loss(y_cv[ok_cv].astype(int), np.tile(np.bincount(y_c[ok_c].astype(int))/ok_c.sum(), (ok_cv.sum(),1))):.3f})")
print("classes:", [wc.COND_CLASSES[int(i)] for i in clf.classes_])

---
## 6 · The final exam: test on 2015–2016 (years the model never saw)

### Baselines — the guesses we must beat
| Baseline | Idea |
|---|---|
| **Persistence** | “Nothing changes: in *h* hours it will be the same as now.” Surprisingly hard to beat for the next few hours. |
| **Same time yesterday** | “It will be like it was at this hour on the latest day we have.” Captures the daily cycle. |
| **Ridge (linear)** | A classic linear model using the same features (one model per horizon). Shows what boosting adds beyond simple maths. |

If our model cannot beat these, it is not worth using. Let's see.

In [ ]:
H_ALL = np.arange(1, wc.MAX_HORIZON + 1)

def truth_all(origin_idx):
    # true values for every start time and every horizon 1..72  ->  {var: array (72, n_origins)}
    return {v: Y[v][origin_idx[None, :] + H_ALL[:, None]] for v in wc.TARGETS}

def predict_all(origin_idx):
    # our model's forecast for every start time and every horizon 1..72  ->  {var: array (72, n_origins)}
    xb, ots = X_base[origin_idx], ts[origin_idx]
    out = {v: np.empty((len(H_ALL), len(origin_idx))) for v in wc.TARGETS}
    for j, h in enumerate(H_ALL):
        M = wc.horizon_matrix(xb, ots, np.full(len(origin_idx), h))
        for v, m in regressors.items():
            spec = wc.TARGETS[v]
            out[v][j] = np.clip(CUR[v][origin_idx] + m.predict(M), spec["lo"], spec["hi"])   # now + predicted change
    return out

def persistence_all(origin_idx):
    return {v: np.tile(CUR[v][origin_idx], (len(H_ALL), 1)) for v in wc.TARGETS}

def yesterday_all(origin_idx):
    k = np.ceil(H_ALL / 24).astype(int)               # go back whole days until we are in the past
    return {v: Y[v][origin_idx[None, :] + H_ALL[:, None] - 24 * k[:, None]] for v in wc.TARGETS}

t0 = time.time()
truth_te, pred_te = truth_all(test_o), predict_all(test_o)
pers_te, yest_te = persistence_all(test_o), yesterday_all(test_o)
print(f"forecasts for {len(test_o):,} test start times x 72 horizons x 6 variables done in {time.time()-t0:.0f}s")

In [ ]:
# Ridge baseline: one linear model per (variable, evaluation horizon)
def ridge_forecast(var, h):
    o = train_o
    Xr = wc.horizon_matrix(X_base[o], ts[o], np.full(len(o), h))
    d = Y[var][o + h] - CUR[var][o]
    ok = ~np.isnan(d)
    m = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge(alpha=10.0)).fit(Xr[ok], d[ok])
    Xt = wc.horizon_matrix(X_base[test_o], ts[test_o], np.full(len(test_o), h))
    return CUR[var][test_o] + m.predict(Xt)

ridge_te = {(v, h): ridge_forecast(v, h) for v in wc.TARGETS for h in wc.EVAL_HORIZONS}

rows = []
for v in wc.TARGETS:
    for h in wc.EVAL_HORIZONS:
        j = h - 1
        preds = {"Persistence": pers_te[v][j], "Same time yesterday": yest_te[v][j],
                 "Ridge (linear)": ridge_te[(v, h)], "Boosting (ours)": pred_te[v][j]}
        for name, p in preds.items():
            rows.append(dict(var=v, horizon=h, model=name, **wc.regression_metrics(truth_te[v][j], p)))
metrics = pd.DataFrame(rows)
print("Metrics computed for", metrics["var"].nunique(), "variables x", metrics["horizon"].nunique(), "horizons x", metrics["model"].nunique(), "models")

### Result 1 — Temperature: average error (MAE, °C) by horizon
Lower is better. Columns are “how many hours ahead”.

In [ ]:
tbl = metrics[metrics["var"] == "temp_c"].pivot(index="model", columns="horizon", values="mae").loc[MODELS]
tbl.columns = [f"+{h} h" for h in tbl.columns]
display(tbl.round(2))

### Result 2 — Skill: how much better than “nothing changes”?
`Skill = 1 − (our error ÷ persistence error)`. **50 % means our error is half of the naive guess.** Negative would mean worse than naive.

In [ ]:
mae = metrics.pivot_table(index=["var", "horizon"], columns="model", values="mae")
skill = (1 - mae["Boosting (ours)"] / mae["Persistence"]).unstack("horizon").loc[list(wc.TARGETS)] * 100
skill.index = [wc.TARGETS[v]["label"] for v in skill.index]
skill.columns = [f"+{h} h" for h in skill.columns]
display(skill.round(0).astype(int).astype(str) + " %")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, v in zip(axes.ravel(), wc.TARGETS):
    curves = {"Persistence": np.nanmean(np.abs(truth_te[v] - pers_te[v]), axis=1),
              "Same time yesterday": np.nanmean(np.abs(truth_te[v] - yest_te[v]), axis=1),
              "Boosting (ours)": np.nanmean(np.abs(truth_te[v] - pred_te[v]), axis=1)}
    for name, c in curves.items():
        ax.plot(H_ALL, c, label=name, **{k: val for k, val in MODEL_STYLE[name].items() if k in ("color", "ls", "lw")})
    rg = metrics[(metrics["var"] == v) & (metrics["model"] == "Ridge (linear)")]
    ax.plot(rg["horizon"], rg["mae"], label="Ridge (linear)", **MODEL_STYLE["Ridge (linear)"])
    ax.set_title(wc.TARGETS[v]["label"], fontsize=11); ax.set_xlabel("hours ahead"); ax.set_ylabel(f"MAE ({wc.TARGETS[v]['unit']})")
    ax.set_xticks([1, 12, 24, 36, 48, 60, 72])
axes[0, 0].legend(frameon=False, fontsize=8)
plt.suptitle("Test years 2015-2016: average error by forecast horizon (lower is better)", y=1.01)
plt.tight_layout(); plt.show()

**What to notice**
* **Persistence** (“nothing changes”) is only competitive for the first hour or two — for wind, pressure and visibility it is on par with our model at +1 h.
* **Boosting** is the most accurate model for temperature, feels-like temperature and humidity from +3 h to about +48 h. The gap is biggest at *medium* horizons (6–24 h), where it combines the daily cycle with the current weather regime.
* **A simple linear model (Ridge) is a strong opponent** — best at +1 h, and for wind and pressure about as good as boosting (slightly better for wind). More complex is not automatically better; that is an honest finding worth mentioning in your viva.
* Persistence's error for temperature wiggles up and down — that is the day/night cycle (guessing "same as now" is worst 12 hours away, when it flips between day and night).
* At +72 h even the best model cannot be perfect: the atmosphere is chaotic. Honest forecasts get *less certain* with time — which is exactly what the uncertainty bands below show.

### Result 3 — R² and RMSE for our model
*R²* = “share of the ups and downs the model explains” (1 = perfect, 0 = as bad as always guessing the average). *RMSE* punishes big misses more than MAE.

In [ ]:
ours = metrics[metrics["model"] == "Boosting (ours)"]
r2 = ours.pivot(index="var", columns="horizon", values="r2").loc[list(wc.TARGETS)]
rmse = ours.pivot(index="var", columns="horizon", values="rmse").loc[list(wc.TARGETS)]
for t_, name in ((r2, "R²  (higher is better)"), (rmse, "RMSE  (lower is better)")):
    t_.index = [f"{wc.TARGETS[v]['label']} ({wc.TARGETS[v]['unit']})" for v in t_.index]
    t_.columns = [f"+{h} h" for h in t_.columns]
    print(name); display(t_.round(2))

---
## 7 · Uncertainty: honest “range” forecasts

A single number (“14 °C”) hides how sure we are. A good forecast says *“14 °C, probably between 12 and 16”*.

**How we build the band (simple and honest):** on the **validation year** we look at how wrong the model was at each horizon.
The 10th and 90th percentile of those errors tell us the typical range. So if at +24 h the model is usually within −2 °C … +2 °C, we draw a band of that size around the forecast.
The band covers about **80 %** of outcomes and gets wider for longer horizons. Then we **check on the test years** that it really does cover ≈ 80 %.

In [ ]:
truth_va, pred_va = truth_all(valid_o), predict_all(valid_o)
intervals = {}
for v in wc.TARGETS:
    err = truth_va[v] - pred_va[v]                                   # (72, n) actual minus forecast
    lo = np.nanpercentile(err, 10, axis=1)
    hi = np.nanpercentile(err, 90, axis=1)
    intervals[v] = dict(lo=wc.smooth(lo, 7), hi=wc.smooth(hi, 7))    # smooth so the band has no jitter

def coverage(truth, pred, lo, hi):
    ok = ~(np.isnan(truth) | np.isnan(pred))
    inside = (truth >= pred + lo[:, None]) & (truth <= pred + hi[:, None])
    return (inside & ok).sum(axis=1) / ok.sum(axis=1)

cov = {v: coverage(truth_te[v], pred_te[v], intervals[v]["lo"], intervals[v]["hi"]) for v in wc.TARGETS}
cov_tbl = pd.DataFrame({wc.TARGETS[v]["label"]: [cov[v][h - 1] * 100 for h in wc.EVAL_HORIZONS] + [cov[v].mean() * 100] for v in wc.TARGETS},
                       index=[f"+{h} h" for h in wc.EVAL_HORIZONS] + ["average"]).T
print("Share of TEST outcomes that fell inside the 80 % band (target = 80):")
display(cov_tbl.round(0).astype(int).astype(str) + " %")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for v in wc.TARGETS:
    ax[0].plot(H_ALL, cov[v] * 100, color=wc.TARGETS[v]["color"], label=wc.TARGETS[v]["label"], lw=1.8)
ax[0].axhline(80, color="k", ls=":"); ax[0].set_ylim(60, 95)
ax[0].set_title("Band coverage on the test years (ideal = 80 %)"); ax[0].set_xlabel("hours ahead"); ax[0].set_ylabel("% of outcomes inside band")
ax[0].legend(frameon=False, fontsize=8, ncol=2)

w = intervals["temp_c"]["hi"] - intervals["temp_c"]["lo"]
ax[1].fill_between(H_ALL, intervals["temp_c"]["lo"], intervals["temp_c"]["hi"], color=wc.TARGETS["temp_c"]["color"], alpha=0.25)
ax[1].axhline(0, color="k", lw=0.8)
ax[1].set_title("Temperature: the band gets wider with time (°C around the forecast)"); ax[1].set_xlabel("hours ahead")
plt.tight_layout(); plt.show()

---
## 8 · Evaluate the sky-condition classifier

Metrics: **accuracy** (how often the top guess is right) and **macro-F1** (average quality across classes, so rare classes like Fog count equally).
Baselines: *persistence* (“the sky stays the same”) and *always guess the most common class*.

In [ ]:
majority = int(np.bincount(COND[train_o].astype(int)).argmax())
rows = []
for h in wc.EVAL_HORIZONS:
    M = wc.horizon_matrix(X_base[test_o], ts[test_o], np.full(len(test_o), h))
    y_true = COND[test_o + h]
    ok = ~np.isnan(y_true)
    P = clf.predict_proba(M)
    y_hat = clf.classes_[P.argmax(axis=1)]
    now = COND[test_o]
    for name, pred in (("Boosting (ours)", y_hat), ("Persistence", now), ("Always most common", np.full(len(test_o), majority))):
        m_ok = ok & ~np.isnan(pred)
        rows.append(dict(horizon=h, model=name, accuracy=accuracy_score(y_true[m_ok], pred[m_ok]),
                         macro_f1=f1_score(y_true[m_ok].astype(int), pred[m_ok].astype(int), average="macro")))
    if h == 24:
        y_true_24, y_hat_24 = y_true, y_hat
cond_metrics = pd.DataFrame(rows)
acc = cond_metrics.pivot(index="model", columns="horizon", values="accuracy").loc[["Always most common", "Persistence", "Boosting (ours)"]]
acc.columns = [f"+{h} h" for h in acc.columns]
print("Accuracy (share of hours where the top guess was right)")
display((acc * 100).round(0).astype(int).astype(str) + " %")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.4), gridspec_kw={"width_ratios": [1, 1]})
for name, sty in (("Always most common", dict(color="#ced4da", ls="--")), ("Persistence", dict(color="#adb5bd", ls="--")), ("Boosting (ours)", dict(color="#d9480f", ls="-", lw=2.6))):
    d = cond_metrics[cond_metrics["model"] == name]
    ax[0].plot(d["horizon"], d["accuracy"] * 100, marker="o", label=name, **sty)
ax[0].set_title("Sky-condition accuracy vs forecast horizon"); ax[0].set_xlabel("hours ahead"); ax[0].set_ylabel("accuracy (%)"); ax[0].legend(frameon=False)

ok = ~np.isnan(y_true_24)
cm = confusion_matrix(y_true_24[ok].astype(int), y_hat_24[ok].astype(int), labels=range(5), normalize="true")
sns.heatmap(cm, annot=True, fmt=".0%", cmap="Blues", ax=ax[1], cbar=False, xticklabels=wc.COND_CLASSES, yticklabels=wc.COND_CLASSES)
ax[1].set_title("+24 h: what really happened (rows) vs what we predicted (columns)"); ax[1].set_ylabel("actual"); ax[1].set_xlabel("predicted")
plt.setp(ax[1].get_xticklabels(), rotation=25, ha="right")
plt.tight_layout(); plt.show()

print(classification_report(y_true_24[ok].astype(int), y_hat_24[ok].astype(int), labels=range(5), target_names=wc.COND_CLASSES, digits=2, zero_division=0))

**What to notice:** predicting *exactly which* cloud category will appear is genuinely hard (Partly vs Mostly Cloudy are neighbours and easily confused),
so accuracy is moderate. Our model beats “always guess the most common class” by a wide margin, roughly ties “the sky stays the same” at short range, and is only modestly ahead of it further out (see the table above).
That is why the app shows *probabilities* rather than a single label — it is more honest about how unsure the model is. Fog gets its own probability chart.

---
## 9 · Explain the model: what does it actually look at?

Machine-learning models should not be black boxes. We use **permutation importance**: shuffle one family of features (e.g. all pressure features) so it carries no
information, and see how much worse the predictions get. Big damage ⇒ the model relied on that family.

In [ ]:
sel = np.random.default_rng(0).choice(len(X_va), 8000, replace=False)   # random rows = all seasons represented
X_imp = X_va[sel]
group_of = np.array([wc.feature_group(c) for c in model_columns])
groups = {g: np.where(group_of == g)[0] for g in np.unique(group_of)}
imp_groups, imp_rng = {}, np.random.default_rng(1)

for v in wc.TARGETS:
    d = Y[v][o_va[sel] + h_va[sel]] - CUR[v][o_va[sel]]
    ok = ~np.isnan(d)
    base = np.abs(regressors[v].predict(X_imp[ok]) - d[ok]).mean()
    scores = {}
    for g, idx in groups.items():
        loss = []
        for _ in range(3):
            Xp = X_imp[ok].copy()
            Xp[:, idx] = Xp[imp_rng.permutation(ok.sum())][:, idx]          # shuffle the whole family together
            loss.append(np.abs(regressors[v].predict(Xp) - d[ok]).mean() - base)
        scores[g] = max(float(np.mean(loss)), 0.0)
    tot = sum(scores.values()) or 1.0
    imp_groups[v] = {g: s / tot for g, s in sorted(scores.items(), key=lambda kv: -kv[1])}

imp_df = pd.DataFrame(imp_groups).fillna(0) * 100
imp_df.columns = [wc.TARGETS[v]["label"] for v in imp_df.columns]
imp_df = imp_df.loc[imp_df.max(axis=1).sort_values(ascending=False).index]
fig, ax = plt.subplots(figsize=(11, 4.8))
sns.heatmap(imp_df, annot=True, fmt=".0f", cmap="YlOrRd", cbar_kws={"label": "% of total importance"}, ax=ax)
ax.set_title("Which information does each forecast rely on? (columns add up to 100 %)"); ax.set_ylabel("")
plt.tight_layout(); plt.show()

**Sanity check with physics** — the model *should* rediscover things meteorologists know: the **forecast horizon and time of day** matter a lot for temperature
(day/night cycle), the **current value of the variable itself** dominates short forecasts, **pressure** trends matter for wind and cloud, and **humidity / dew-point spread** matter for visibility (fog).

In [ ]:
s4 = sel[:4000]
d = Y["temp_c"][o_va[s4] + h_va[s4]] - CUR["temp_c"][o_va[s4]]
ok = ~np.isnan(d)
pi = permutation_importance(regressors["temp_c"], X_va[s4][ok], d[ok], n_repeats=2, random_state=0,
                            scoring="neg_mean_absolute_error", n_jobs=1)
top = pd.Series(pi.importances_mean, index=model_columns).sort_values(ascending=False).head(15)
fig, ax = plt.subplots(figsize=(9, 4.6))
top[::-1].plot(kind="barh", color=wc.TARGETS["temp_c"]["color"], ax=ax)
ax.set_title("Temperature model: 15 most important individual features"); ax.set_xlabel("increase in MAE (°C) when the feature is shuffled")
plt.tight_layout(); plt.show()
top_features_temp = [(k, float(v)) for k, v in top.items()]

---
## 10 · Save everything for the app

We write into `models/`:
* `regressors.joblib` — the 6 weather models,
* `classifier.joblib` — the sky-condition model,
* `meta.json` — feature names, uncertainty bands, all the scores and charts data the app's *Model Insights* page shows,
* two `.csv` files with the metrics (handy for your report).

In [ ]:
joblib.dump(regressors, MODEL_DIR / "regressors.joblib", compress=3)
joblib.dump(clf, MODEL_DIR / "classifier.joblib", compress=3)

curves = {v: {"Persistence": np.nanmean(np.abs(truth_te[v] - pers_te[v]), axis=1).tolist(),
              "Same time yesterday": np.nanmean(np.abs(truth_te[v] - yest_te[v]), axis=1).tolist(),
              "Boosting (ours)": np.nanmean(np.abs(truth_te[v] - pred_te[v]), axis=1).tolist()} for v in wc.TARGETS}

okk = ~np.isnan(y_true_24)
cm24 = confusion_matrix(y_true_24[okk].astype(int), y_hat_24[okk].astype(int), labels=range(5), normalize="true")
rep24 = classification_report(y_true_24[okk].astype(int), y_hat_24[okk].astype(int), labels=range(5),
                              target_names=wc.COND_CLASSES, output_dict=True, zero_division=0)

meta = dict(
    created_at=pd.Timestamp.now().isoformat(timespec="seconds"),
    versions=dict(python=platform.python_version(), **{"scikit-learn": sklearn.__version__, "numpy": np.__version__, "pandas": pd.__version__}),
    base_columns=base_columns, model_columns=model_columns,
    max_horizon=wc.MAX_HORIZON, eval_horizons=wc.EVAL_HORIZONS, interval_level=wc.INTERVAL_LEVEL,
    cond_classes=wc.COND_CLASSES,
    data=dict(rows=len(df), start=str(df.index.min()), end=str(df.index.max()), cleaning_report=report),
    splits=dict(train=[str(ts[train_o[0]]), str(ts[train_o[-1]])], valid=[str(ts[valid_o[0]]), str(ts[valid_o[-1]])],
                test=[str(ts[test_o[0]]), str(ts[test_o[-1]])], n_train_starts=len(train_o), n_valid_starts=len(valid_o), n_test_starts=len(test_o)),
    hyperparameters=dict(regressor=dict(HGB_PARAMS, max_iter_limit=MAX_ITER_REG, rows_per_origin=ROWS_PER_ORIGIN_REG, target_scaling="change / typical change at that horizon"),
                         classifier=dict(CLF_PARAMS, max_iter_limit=MAX_ITER_CLF, rows_per_origin=ROWS_PER_ORIGIN_CLF)),
    best_iterations=dict(best_iter, classifier=clf_best),
    intervals={v: dict(lo=intervals[v]["lo"].tolist(), hi=intervals[v]["hi"].tolist()) for v in wc.TARGETS},
    coverage={v: cov[v].tolist() for v in wc.TARGETS},
    metrics=metrics.to_dict("records"),
    curves=curves,
    condition_metrics=cond_metrics.to_dict("records"),
    condition_confusion_24h=cm24.tolist(),
    condition_report_24h=rep24,
    importance_groups=imp_groups,
    top_features_temp=top_features_temp,
)
(MODEL_DIR / "meta.json").write_text(json.dumps(meta, indent=1, default=lambda o: o.item() if hasattr(o, "item") else str(o)), encoding="utf-8")
metrics.round(4).to_csv(MODEL_DIR / "metrics_regression.csv", index=False)
cond_metrics.round(4).to_csv(MODEL_DIR / "metrics_condition.csv", index=False)

for p in sorted(MODEL_DIR.iterdir()):
    print(f"{p.name:<26}{p.stat().st_size/1e6:>8.2f} MB")

---
## 11 · End-to-end demo (loads the *saved* files, exactly like the app)

This proves the saved models work: we load them from disk, forecast 72 hours from a random moment in the test years, and compare with what really happened.

In [ ]:
bundle = wc.load_bundle(MODEL_DIR)
origin = ts[test_o[np.argmin(np.abs((ts[test_o] - pd.Timestamp("2016-07-14 09:00", tz="UTC")).total_seconds()))]]
fc, now = wc.make_forecast(feat, origin, bundle, horizon=72)
actual = df.loc[fc.index]

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for ax, v in zip(axes, ["temp_c", "hum", "wind_kmh"]):
    x = wc.to_local(fc.index)
    ax.fill_between(x, fc[f"{v}_lo"], fc[f"{v}_hi"], color=wc.TARGETS[v]["color"], alpha=0.2, label="80 % band")
    ax.plot(x, fc[v], color=wc.TARGETS[v]["color"], lw=2.4, label="forecast")
    ax.plot(x, actual[v], color="k", lw=1.2, label="what really happened")
    ax.set_title(f"{wc.TARGETS[v]['label']} ({wc.TARGETS[v]['unit']})"); plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
axes[0].legend(frameon=False, fontsize=8)
plt.suptitle(f"72-hour forecast made at {wc.to_local(pd.DatetimeIndex([origin]))[0]:%Y-%m-%d %H:%M} (local time)", y=1.03)
plt.tight_layout(); plt.show()
display(fc[["temp_c", "hum", "wind_kmh", "pres_mb", "cond", "cond_conf"]].iloc[[0, 5, 11, 23, 47, 71]].round(2))

In [ ]:
# ───────────────  Scoreboard  ───────────────
t = metrics[metrics["var"] == "temp_c"].pivot(index="model", columns="horizon", values="mae")
print("HEADLINE RESULTS (test years 2015-2016, never seen in training)")
print("-" * 68)
for h in (6, 24, 72):
    ours_, naive_ = t.loc["Boosting (ours)", h], t.loc["Persistence", h]
    print(f"Temperature +{h:>2} h : our error {ours_:.2f} °C  vs  'nothing changes' {naive_:.2f} °C   ->  {100*(1-ours_/naive_):.0f} % better")
print(f"80 % bands actually covered {np.mean([cov[v].mean() for v in wc.TARGETS])*100:.0f} % of outcomes on the test years")
a24 = cond_metrics[(cond_metrics.horizon == 24)].set_index("model")["accuracy"]
print(f"Sky condition +24 h : accuracy {a24['Boosting (ours)']*100:.0f} %  vs  persistence {a24['Persistence']*100:.0f} %  vs  most-common-class {a24['Always most common']*100:.0f} %")

---
## 12 · Summary, limits and next steps

**What we built:** a direct multi-horizon forecaster (1–72 h) for temperature, feels-like temperature, humidity, wind, pressure and visibility, with calibrated 80 % bands and a sky-condition probability model,
validated on two unseen years against three baselines, with leakage tests and an explanation of what drives the forecasts.

**Honest limits (say these in your viva — they show maturity)**
* The data comes from **one weather station** only, so the model does not know about weather approaching from other places. Real forecasts use satellites and physics-based models (NWP); ours learns from one station's history, which is why skill fades after ~1–2 days.
* The dataset has **no rainfall amount or cloud-cover measurement** (`Loud Cover` is all zeros; `Precip Type` is essentially “snow if < 0 °C”). So we can forecast *sky condition*, but not “chance of rain”.
* The uncertainty band width is calibrated on one year (2014) and is the same in all seasons. On the test years the bands come out slightly too narrow for temperature and pressure (about 78 % coverage instead of 80 %).
* Data ends on 31 Dec 2016 — the app is a demonstration on historical data, not a live service.

**Ideas for the next level (ask me if you want any of these built):** LSTM / Transformer models, adding live data from a weather API, several cities, seasonal-aware bands, SHAP explanations, model comparison with XGBoost/LightGBM/CatBoost.